# Vanguard_AB 

# Day 02 - Cleaning & Merging datasets by CLIENT_ID

# First EDAs and Customer Demographics Overviews -- EDA/Cleanup notes D01 

## Data Sources
(1)     Client Profiles -- User demographics (df_final_demo.txt)
(70610, 9: client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth) 
(2)     Digital Footprints -- User website interactions 
2.1 Part1 (df_final_web_data_pt_1.txt)
(343142, 5: client_id,visitor_id,visit_id,process_step,date_time) 
2.2 Part2 (df_final_web_data_pt_2.txt)
(412265, 5: client_id,visitor_id,visit_id,process_step,date_time) 
(3)     Experiment Roster -- Control or Test Group (df_final_experiment_clients.txt)
(70610, 2: client_id, Variation) 

## TODOS
--> Cleanup 1. and 3. - L & AD
--> Merge 1 & 3 into one DE-anonymized Client profiles list - ALL, P for colab nb
(Connect by client_ID, Join 9 ClientProfiles columns (or drop some?), PLUS the A or B check col Testgroup Y/N)
--> Store Merged DE-Anon Client Profiles in pandas df (PLUS csv? parquet? sql db?) - TODO

--> First Analysis of A-B Groups:
Simple Demographics (eg, Avg age, gender, location, tenure, site visits?) - TODO
Common sense checks (Differences between Test group A and Control B? Skews and Bias? Sample sizes comparable? Variances similar?)
--> Make ER diagram of the db relations (with client_id as connector) - TODO
(Keep (2) and (1&3) as distinct tables for now -- no need to merge them YET! We have client_id as connecting Key)

# First EDAs and Customer Demographics Overviews 
EDA/Cleanup notes D01 --> See trello L.

**Safe Import & Table Comparison/Verification** 
Two Files on "Digital Footprints" (User website interactions) 
Part1 (343143, 5) & Part2 (412266, 5); both have columns (client_id,visitor_id,visit_id,process_step,date_time) 
--> CHECK IF THEY ARE IDENTICAL AND CAN BE MERGED

# CAVEAT
A common trap in A/B testing data is Encoding or Data Type mismatches (like ID var being Int in one table and Str elsewhere, like repeated user actions, server errors and bugs etc.)

# Check immediately AFTER merging:
-Dtypes - eg. check if date_time is Type DATETIME or set to Datetime if imported as string --> df_web_data.info() 
-Duplicates - AB test logs have LOTS of duplicates (like when user refreshed page) --> df_web_data.duplicated().sum()
-Timeline - get min/max of the date_time column, is there a time gap between part1 & 2, or doubled dates/overlaps?


# Load & Merge ds 2.1 and 2.2 (Footprints)

- Merge 2.1 & 2.2 into one df df_final_web_data 
- Cleanup datetime col and Save merged df as Parquet (or csv)

In [ ]:
# 1. First Look at web_data in Pandas
# Compare both txt tables and check for swapped columns or different param.
import pandas as pd

# Peek at first 5 rows 
path1 = 'df_final_web_data_pt_1.txt'
path2 = 'df_final_web_data_pt_2.txt'

# Get structure using nrows=5
df1_peek = pd.read_csv(path1, sep=',', nrows=5) 
df2_peek = pd.read_csv(path2, sep=',', nrows=5)

print(df1_peek.columns)
print(df2_peek.columns)

# Check if columns match in terms of name and order
print(f"Columns match: {list(df1_peek.columns) == list(df2_peek.columns)}")

# 2. Load and concatenate txt files 
# Needs ca. 200-500MB of RAM to load both in Pandas (almost 1 Mio. lines)
# Execute in a single notebook and refresh before, don't keep lots of copies in memory! 

# Load both files as dfs 
df1 = pd.read_csv(path1)
df2 = pd.read_csv(path2)

# Vertical Concat into ONE main df_web_data
df_web_data = pd.concat([df1, df2], axis=0, ignore_index=True)  

# Clean memory -- Delete the two partial dfs right away to prevent overload!
del df1
del df2


Index(['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time'], dtype='object')
Index(['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time'], dtype='object')
Columns match: True


In [ ]:
# 3. Column Verification
df_web_data.info()
df_web_data.duplicated().sum()

# Convert date_time to datetime objects
df_web_data['date_time'] = pd.to_datetime(df_web_data['date_time'])

# 4. Store as compressed Parquet file to Preserve DTypes/Formatting 'myfile.parquet'
# Don't use the CSV or txt files for this - save with Pandas method df.to_parquet() 

# Save as Parquet
df_web_data.to_parquet('merged_web_data.parquet')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 755405 entries, 0 to 755404
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   client_id     755405 non-null  int64 
 1   visitor_id    755405 non-null  object
 2   visit_id      755405 non-null  object
 3   process_step  755405 non-null  object
 4   date_time     755405 non-null  object
dtypes: int64(1), object(4)
memory usage: 28.8+ MB


In [ ]:
# 5. Upload to local mySQL Database if needed to do joins, filter queries etc. -- TODO 
# DON'T push merged web_data csv or parquet file to Github -- use locally and add to gitignore -- TODO

# 6. JOIN to other two datasets via client_id -- TODO

# 2. Load Cleanup & Merge ds 1 and 3 
Client Profiles (final_demo) and Roster (experiment_clients) 

--> Merge 1 & 3 into one DE-anonymized Client profiles list 
(Connect by client_ID, Join 9 ClientProfiles columns (or drop some?), PLUS the A or B check col "VARIATION" for Testgroup Y/N)

--> Store Merged DE-Anon Client Profiles in pandas df (PLUS csv? parquet? sql db?) - TODO



# 3. First Analysis of A-B Groups:

--> Simple Demographics (eg, Avg age, gender, location, tenure, site visits?) - DONE L
--> Common sense checks (Differences between Test group A and Control B? Skews and Bias? Sample sizes comparable? Variances similar?) - TODO
--> Make ER diagram of the db relations (with client_id as connector) - TODO

(Keep (2) and (1&3) as distinct tables for now! We have client_id as connecting Key)